# EDA

En esta fase analizamos los datos ya cargados para entender su estructura antes de pasar al modelo.
Para esto se calculan estadísticas generales, se revisa el balance de clases en las máscaras, la distribución de intensidades, y se visualizan algunas muestras.

1. Resumen general: tamaño de imágenes, tipos de dato y rangos de valores.
1. Balance de clases: qué proporción de píxeles corresponde a cada tipo de hallazgo en la máscara.
1. Distribución de intensidad: cómo se distribuyen los valores de píxel (en HU).
1. Área de lesión por imagen: no solo el promedio global, sino qué tanto varía el tamaño de la lesión entre muestras.
1. Componentes conectados: cuántas regiones separadas de lesión suele haber por imagen.
1. Visualización: comparar imagen y máscara lado a lado para confirmar que están bien alineadas.

## Hallazgos

- Las imágenes vienen en Unidades Hounsfield (HU), no en escala 0-255 como una foto normal (rango observado: -1606 a 597 aproximadamente). Esto hay que tomarlo en cuenta al normalizar para el modelo.
- Las máscaras son one-hot con 4 canales, en este orden: `ground_glass`, `consolidations`, `lungs_other`, `background`.
- Hay un desbalance fuerte de clases: el fondo ocupa la mayoría de los píxeles (~73.6%), seguido de tejido pulmonar sano (~19.1%), y las lesiones son la minoría (ground-glass ~4.6%, consolidación ~2.6%). Esto es importante para cuando se elija la función de pérdida del modelo.
- En el split de entrenamiento no hay imágenes completamente sin lesión (0/80), lo que sugiere que este subconjunto ya viene filtrado a solo casos positivos.
- El área de lesión por imagen varía bastante: en promedio ocupa 7.29% de la imagen (mediana 5.23%), pero hay casos de hasta 25.90%.
- En promedio hay ~7.65 regiones separadas de lesión por imagen (máximo 34), lo que indica que las lesiones suelen aparecer como varios focos

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def summarize_dataset(images, masks, name=""):
    print(f"--- {name} ---")
    print(f"Imágenes: {images.shape}, dtype={images.dtype}, min={images.min():.3f}, max={images.max():.3f}")
    print(f"Máscaras: {masks.shape}, dtype={masks.dtype}, min={masks.min():.3f}, max={masks.max():.3f}")
    n_classes = masks.shape[-1] if masks.ndim == 4 else 1
    print(f"Número de clases en máscara: {n_classes}")

In [ ]:
def plot_class_balance(masks, class_names=None):
    """
    Proporción de píxeles por clase
    """
    n_classes = masks.shape[-1]
    if class_names is None:
        class_names = [f"clase_{i}" for i in range(n_classes)]

    pixel_counts = masks.sum(axis=(0, 1, 2))
    proportions = pixel_counts / pixel_counts.sum()

    plt.figure(figsize=(6, 4))
    plt.bar(class_names, proportions)
    plt.ylabel("Proporción de píxeles")
    plt.title("Balance de clases en las máscaras")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    for name, prop in zip(class_names, proportions):
        print(f"{name}: {prop:.4%}")
        
    return proportions

In [ ]:
def count_empty_masks(masks, background_idx=0):
    """
    Cuenta cuántas imágenes no tienen lesión (100% fondo).
    """
    n_classes = masks.shape[-1]
    lesion_channels = [i for i in range(n_classes) if i != background_idx]
    has_lesion = masks[..., lesion_channels].sum(axis=(1, 2, 3)) > 0

    n_empty = (~has_lesion).sum()
    n_total = len(masks)
    print(f"Imágenes sin lesión: {n_empty}/{n_total} ({n_empty/n_total:.1%})")
    return has_lesion

In [ ]:
def visualize(image_batch, mask_batch=None, pred_batch=None, num_samples=8, hot_encode=True):
    """Muestra imagen + cada canal de máscara (y predicción opcional) lado a lado."""
    num_classes = mask_batch.shape[-1] if mask_batch is not None else 0
    fig, ax = plt.subplots(num_classes + 1, num_samples, figsize=(num_samples * 2, (num_classes + 1) * 2))

    for i in range(num_samples):
        ax_image = ax[0, i] if num_classes > 0 else ax[i]
        if hot_encode:
            ax_image.imshow(image_batch[i, :, :, 0], cmap='Greys')
        else:
            ax_image.imshow(image_batch[i, :, :])
        ax_image.set_xticks([])
        ax_image.set_yticks([])

        if mask_batch is not None:
            for j in range(num_classes):
                if pred_batch is None:
                    mask_to_show = mask_batch[i, :, :, j]
                else:
                    mask_to_show = np.zeros(shape=(*mask_batch.shape[1:-1], 3))
                    mask_to_show[..., 0] = pred_batch[i, :, :, j] > 0.5
                    mask_to_show[..., 1] = mask_batch[i, :, :, j]
                ax[j + 1, i].imshow(mask_to_show, vmin=0, vmax=1)
                ax[j + 1, i].set_xticks([])
                ax[j + 1, i].set_yticks([])

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_hists(images1, images2=None, label1="dataset 1", label2="dataset 2"):
    """
    Compara distribución de intensidades de píxel entre dos datasets
    """
    plt.hist(images1.ravel(), bins=100, density=True, color='b', alpha=1 if images2 is None else 0.5, label=label1)
    if images2 is not None:
        plt.hist(images2.ravel(), bins=100, density=True, alpha=0.5, color='orange', label=label2)
        plt.legend()
    plt.title("Distribución de intensidades" if images2 is None else "Comparación de distribuciones")
    plt.xlabel("Valor de píxel")
    plt.ylabel("Densidad")
    plt.show()

In [ ]:
def plot_intensity_distribution(images, sample_size=500):
    """Histograma de intensidades de píxel (formato HU), sobre una muestra."""
    sample = images[:sample_size].flatten()
    plt.figure(figsize=(6, 4))
    plt.hist(sample, bins=50)
    plt.title("Distribución de intensidad de píxeles")
    plt.xlabel("Valor de píxel (HU)")
    plt.ylabel("Frecuencia")
    plt.show()

In [ ]:
def plot_lesion_area_per_image(masks, lesion_channels, background_idx=None):
    """
    Histograma del % de área con lesión, por imagen (no el promedio global).
    Muestra qué tan parejo o disperso es el tamaño de lesión entre muestras.
    """
    lesion_area = masks[..., lesion_channels].sum(axis=(1, 2, 3))
    total_area = masks.shape[1] * masks.shape[2]
    lesion_pct = lesion_area / total_area * 100

    plt.figure(figsize=(6, 4))
    plt.hist(lesion_pct, bins=30)
    plt.title("% de área con lesión por imagen")
    plt.xlabel("% del área total")
    plt.ylabel("Número de imágenes")
    plt.show()

    print(f"Media: {lesion_pct.mean():.2f}%, mediana: {np.median(lesion_pct):.2f}%, max: {lesion_pct.max():.2f}%")
    return lesion_pct


In [ ]:
def count_connected_components(masks, lesion_channels):
    """
    Cuenta cuántas regiones separadas de lesión hay por imagen
    (usa scipy.ndimage.label). Útil para saber si suelen ser lesiones
    aisladas o múltiples focos dispersos.
    """
    from scipy import ndimage

    lesion_mask = masks[..., lesion_channels].sum(axis=-1) > 0
    counts = []
    for i in range(len(masks)):
        _, n_components = ndimage.label(lesion_mask[i])
        counts.append(n_components)

    counts = np.array(counts)
    print(f"Componentes por imagen -> media: {counts.mean():.2f}, max: {counts.max()}")
    return counts

In [ ]:
def run_eda(images, masks, name="train", class_names=None, background_idx=3, lesion_channels=None):
    if lesion_channels is None:
        lesion_channels = [0, 1]  # ground_glass, consolidations (excluye lungs_other y background)

    summarize_dataset(images, masks, name=name)

    has_lesion = count_empty_masks(masks, background_idx=background_idx)
    proportions = plot_class_balance(masks, class_names=class_names)

    plot_intensity_distribution(images)
    lesion_pct = plot_lesion_area_per_image(masks, lesion_channels=lesion_channels)
    n_components = count_connected_components(masks, lesion_channels=lesion_channels)

    visualize(images[:8], masks[:8])

    return {
        "n_samples": len(images),
        "has_lesion": has_lesion,
        "class_proportions": proportions,
        "lesion_pct_per_image": lesion_pct,
        "n_components_per_image": n_components,
    }